# Zero-sum games and minimax

A lesson from the Python and AI School, by Dr. Behrooz Taheri

In a **zero-sum** game, one player's gain is the other's loss. Von Neumann showed that with mixed strategies the game has a unique value:

$$\max_p\min_q\ p^TAq=\min_q\max_p\ p^TAq$$

## Example
For $A=\begin{bmatrix}3&-1\\-2&4\end{bmatrix}$, equating the two columns' payoffs gives $3p-2(1-p)=-p+4(1-p)$, so $p=0.6$ and the value is 1.

Linear programming also solves it:



In [ ]:
import numpy as np
from scipy.optimize import linprog

def solve_zero_sum(A):
    A = np.array(A, float)
    m, n = A.shape
    c = [0] * m + [-1]                  # maximize the value v
    A_ub = np.c_[-A.T, np.ones(n)]
    A_eq = [[1] * m + [0]]
    r = linprog(c, A_ub=A_ub, b_ub=np.zeros(n), A_eq=A_eq, b_eq=[1], bounds=[(0, 1)] * m + [(None, None)])
    return r.x[:m].round(3), round(r.x[-1], 3)

print(solve_zero_sum([[3, -1], [-2, 4]]))
print("Rock-paper-scissors:", solve_zero_sum([[0, -1, 1], [1, 0, -1], [-1, 1, 0]]))



## Minimax search in games
In turn-based games such as tic-tac-toe, the state tree is searched. **Alpha-beta pruning** removes useless branches.



In [ ]:
from functools import lru_cache

LINES = [(0,1,2),(3,4,5),(6,7,8),(0,3,6),(1,4,7),(2,5,8),(0,4,8),(2,4,6)]

def winner(b):
    for i, j, k in LINES:
        if b[i] != "." and b[i] == b[j] == b[k]:
            return b[i]
    return None

@lru_cache(maxsize=None)
def value(b, player):
    w = winner(b)
    if w:
        return 1 if w == "X" else -1
    if "." not in b:
        return 0
    other = "O" if player == "X" else "X"
    scores = [value(b[:i] + player + b[i+1:], other) for i, c in enumerate(b) if c == "."]
    return max(scores) if player == "X" else min(scores)

print("Value of tic-tac-toe with perfect play:", value("." * 9, "X"))